In [1]:
from modelscope import snapshot_download
import os

os.environ["MODELSCOPE_CACHE"] = "./.cache"
os.environ["MODELSCOPE_ENDPOINT"] = "https://modelscope.cn"

model_dir = snapshot_download(
    "qwen/Qwen2.5-7B-Instruct",
    local_dir="./qwen/Qwen2.5-7B-Instruct",   
    revision="master"
)

print("✅ 下载完成！模型路径：", model_dir)

2026-03-26 10:49:39,332 - modelscope - WARNING - Using branch: master as version is unstable, use with caution
2026-03-26 10:49:39,430 - modelscope - INFO - Target directory already exists, skipping creation.


✅ 下载完成！模型路径： ./qwen/Qwen2.5-7B-Instruct


BIO数据
  ↓
PeopleDaily（解析为 sentence + span）
  ↓
to_hf（变成HF Dataset）
  ↓
encode_batch 
  ↓
token-level输入（input_ids + labels）
  ↓
模型训练（Token Classification）

In [2]:
from torch.utils.data import Dataset

categories = set()

class PeopleDaily(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f.read().split('\n\n')):
                if not line:
                    break
                sentence, labels = '', []
                for i, item in enumerate(line.split('\n')):
                    char, tag = item.split(' ')
                    sentence += char
                    if tag.startswith('B'):
                        labels.append([i, i, char, tag[2:]]) 
                        categories.add(tag[2:])
                    elif tag.startswith('I'):
                        labels[-1][1] = i
                        labels[-1][2] += char
                Data[idx] = {
                    'sentence': sentence, 
                    'labels': labels
                }
        return Data
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]
train_data = PeopleDaily('/t9k/mnt/lxq/FineTuning/IdentifyHerb/data/raw/medical.train')
valid_data = PeopleDaily('/t9k/mnt/lxq/FineTuning/IdentifyHerb/data/raw/medical.dev')
test_data = PeopleDaily('/t9k/mnt/lxq/FineTuning/IdentifyHerb/data/raw/medical.test')  

id2label = {0:'O'}
for c in list(sorted(categories)):
    id2label[len(id2label)] = f"B-{c}"
    id2label[len(id2label)] = f"I-{c}"
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)



In [3]:
import json
from datasets import Dataset as HFDataset
def to_hf(pd_dataset):
    sentences, labels = [], []
    for i in range(len(pd_dataset)):
        sentences.append(pd_dataset[i]['sentence'])   
        labels.append(json.dumps(pd_dataset[i]['labels'], ensure_ascii=False))
    return HFDataset.from_dict({'sentence': sentences, 'labels': labels})

train_ds = to_hf(train_data)
eval_ds  = to_hf(valid_data)
test_ds  = to_hf(test_data)

In [4]:
import torch
from torch.utils.data import DataLoader
from modelscope import AutoTokenizer,AutoModelForTokenClassification
import numpy as np

model_dir = "./qwen/Qwen2.5-7B-Instruct"  
tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True, trust_remote_code=True)
def encode_batch(examples):
    sents   = examples["sentence"]
    lb_strs = examples["labels"]

    
    encoded = tokenizer(sents, padding=False, truncation=True)

    
    all_labels = []
    for b_idx, (sent, lb_str) in enumerate(zip(sents, lb_strs)):
        lbs = json.loads(lb_str)
        enc = tokenizer(sent, add_special_tokens=True)   
        n   = len(enc.input_ids)
        labels = [-100] * n

        
        for char_start, char_end, _, tag in lbs:
            tok_start = None; tok_end = None
            for t_idx, tok in enumerate(enc.tokens()):
                tok_s = len(tokenizer.decode(enc.input_ids[:t_idx], skip_special_tokens=True))
                tok_e = len(tokenizer.decode(enc.input_ids[:t_idx+1], skip_special_tokens=True))
                if tok_s == char_start: tok_start = t_idx
                if tok_e == char_end + 1: tok_end = t_idx
                if tok_start is not None and tok_end is not None: break
            if tok_start is None or tok_end is None or tok_end < tok_start:
                continue
            labels[tok_start] = label2id[f"B-{tag}"]
            if tok_end > tok_start:
                labels[tok_start+1:tok_end+1] = [label2id[f"I-{tag}"]] * (tok_end - tok_start)
        all_labels.append(labels)

    
    return {"input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "labels": all_labels}


train_tensor = train_ds.map(encode_batch, batched=True, remove_columns=["sentence", "labels"])
eval_tensor  = eval_ds.map(encode_batch,  batched=True, remove_columns=["sentence", "labels"])
test_tensor  = test_ds.map(encode_batch,  batched=True, remove_columns=["sentence", "labels"])

Map:   0%|          | 0/5259 [00:00<?, ? examples/s]

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

Map:   0%|          | 0/658 [00:00<?, ? examples/s]

In [5]:
print(train_tensor[0])
print(id2label)
print(label2id)

{'labels': [-100, -100, -100, 1, 2], 'input_ids': [46451, 64355, 101616, 39426, 99746], 'attention_mask': [1, 1, 1, 1, 1]}
{0: 'O', 1: 'B-c', 2: 'I-c', 3: 'B-中医治则', 4: 'I-中医治则', 5: 'B-中医治疗', 6: 'I-中医治疗', 7: 'B-中医证候', 8: 'I-中医证候', 9: 'B-中医诊断', 10: 'I-中医诊断', 11: 'B-中药', 12: 'I-中药', 13: 'B-临床表现', 14: 'I-临床表现', 15: 'B-其他治疗', 16: 'I-其他治疗', 17: 'B-方剂', 18: 'I-方剂', 19: 'B-西医治疗', 20: 'I-西医治疗', 21: 'B-西医诊断', 22: 'I-西医诊断'}
{'O': 0, 'B-c': 1, 'I-c': 2, 'B-中医治则': 3, 'I-中医治则': 4, 'B-中医治疗': 5, 'I-中医治疗': 6, 'B-中医证候': 7, 'I-中医证候': 8, 'B-中医诊断': 9, 'I-中医诊断': 10, 'B-中药': 11, 'I-中药': 12, 'B-临床表现': 13, 'I-临床表现': 14, 'B-其他治疗': 15, 'I-其他治疗': 16, 'B-方剂': 17, 'I-方剂': 18, 'B-西医治疗': 19, 'I-西医治疗': 20, 'B-西医诊断': 21, 'I-西医诊断': 22}


In [6]:
from peft import LoraConfig, get_peft_model, TaskType
model = AutoModelForTokenClassification.from_pretrained(model_dir, num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    device_map="auto", 
    torch_dtype=torch.bfloat16)
model.enable_input_require_grads()  
config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=False,  
    r=8,  
    lora_alpha=16,  
    lora_dropout=0.1,  
)
device = 'cuda'
model = get_peft_model(model, config).to(device)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of Qwen2ForTokenClassification were not initialized from the model checkpoint at ./qwen/Qwen2.5-7B-Instruct and are newly initialized: ['score.bias', 'score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    
    true_ent = []; pred_ent = []
    for label, pred in zip(labels, preds):
        mask = label != -100
        label = label[mask]
        pred  = pred[mask]

        
        def extract_entities(ids):
            entities = []
            start = 0
            while start < len(ids):
                lbl = id2label[ids[start]]
                if lbl.startswith("B-"):
                    ent_type = lbl[2:]
                    end = start + 1
                    while end < len(ids) and id2label[ids[end]] == f"I-{ent_type}":
                        end += 1
                    entities.append((start, end - 1, ent_type))
                    start = end
                else:
                    start += 1
            return entities

        true_ent.extend(extract_entities(label))
        pred_ent.extend(extract_entities(pred))

    
    entities_true = {} 
    entities_pred = {}
    for start, end, ent_type in true_ent:
        entities_true.setdefault(ent_type, []).append((start, end))
    for start, end, ent_type in pred_ent:
        entities_pred.setdefault(ent_type, []).append((start, end))

    
    out = {"macro_f1": 0.0}
    all_f1 = []
    print("\n========== 实体级 F1（严格匹配） ==========")
    for ent_type in sorted(set(entities_true.keys()) | set(entities_pred.keys())):
        y_true = set(entities_true.get(ent_type, []))
        y_pred = set(entities_pred.get(ent_type, []))
        tp = len(y_true & y_pred)
        fp = len(y_pred - y_true)
        fn = len(y_true - y_pred)
        f1 = tp / (tp + 0.5 * (fp + fn)) if (tp + fp + fn) > 0 else 0.0
        all_f1.append(f1)
        print(f"{ent_type:<25}: {f1:.4f}")
        out[f"{ent_type}_f1"] = f1

    macro = np.mean(all_f1) if all_f1 else 0.0
    out["macro_f1"] = macro
    print(f"\nMacro-F1: {macro:.4f}", "="*40)
    return out

In [ ]:
from transformers import TrainingArguments, Trainer,DefaultDataCollator,DataCollatorForTokenClassification
dummy_collator = DefaultDataCollator(return_tensors='pt')
args = TrainingArguments(
    output_dir="./model_lora",
        per_device_train_batch_size=4,  
        gradient_accumulation_steps=4,  
        logging_steps=10,                
        num_train_epochs=3,    
        eval_strategy="steps",  
        eval_steps=100, 
        save_steps=100, 
        learning_rate=1e-4,
        save_on_each_node=True,
        load_best_model_at_end=True, 
        gradient_checkpointing=True  
    )
trainer = Trainer(
    model=model, 
    args=args,
    train_dataset=train_tensor,          
    eval_dataset=eval_tensor,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_2764051/2632179359.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
No label_names provided for model class `PeftModelForTokenClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [9]:
print(train_ds is trainer.train_dataset)
for batch in trainer.get_train_dataloader():
    print("keys:", list(batch.keys()))
    break

False
keys: ['input_ids', 'attention_mask', 'labels']


In [10]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Output()

Output()

swanlab: Tracking run with swanlab version 0.7.13

swanlab: Run data will be saved locally in 
/t9k/mnt/lxq/FineTuning/IdentifyHerb/swanlog/run-20260326_105011-fw4hggpgobhiyxfu7k1qs

...

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f25a41d5810>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f27078ffa30, execution_count=10 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f27078ffa00, raw_cell="trainer.train()" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a2249424d43227d/t9k/mnt/lxq/FineTuning/IdentifyHerb/ner_qwen_lora.ipynb#X13sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

swanlab: Error executing task: Run directory does not exist when accessing media directory.

swanlab: Error executing task: Run directory does not exist when accessing media directory.

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_tensor)
print("\n========== 测试集 实体级 F1 ==========")
for k, v in test_metrics.items():
    if k.endswith("_f1"):
        print(f"{k:<25}: {v:.4f}")


macro = test_metrics["eval_macro_f1"]
print(f"\n测试集 Macro-F1: {macro:.4f}")


========== 实体级 F1（严格匹配） ==========
中医治则                     : 0.8235
中医治疗                     : 0.8667
中医证候                     : 0.8471
中医诊断                     : 0.8000
中药                       : 0.8905
临床表现                     : 0.9009
其他治疗                     : 0.9333
方剂                       : 0.8000
西医治疗                     : 0.9268
西医诊断                     : 0.9280

Macro-F1: 0.8717 ========================================

========== 测试集 实体级 F1 ==========
eval_macro_f1            : 0.8717
eval_中医治则_f1             : 0.8235
eval_中医治疗_f1             : 0.8667
eval_中医证候_f1             : 0.8471
eval_中医诊断_f1             : 0.8000
eval_中药_f1               : 0.8905
eval_临床表现_f1             : 0.9009
eval_其他治疗_f1             : 0.9333
eval_方剂_f1               : 0.8000
eval_西医治疗_f1             : 0.9268
eval_西医诊断_f1             : 0.9280

测试集 Macro-F1: 0.8717
